In [1]:
import json
import numpy as np

soft_label_thresh = 0.5

# COCO

In [2]:
# load dataset with soft labels and coco original gt
dataset = "COCO"
with open(f"../our_datasets/{dataset}.json") as f:
    data = json.load(f)


with open("/path/to/datasets/COCO/2017/annotations/instances_val2017.json") as f:
    coco_gt = json.load(f)

filename_to_id = {
    img["file_name"]: img["id"]
    for img in coco_gt["images"]
}

id_map = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 10, 10: 11, 11: 13, 12: 14, 13: 15, 14: 16, 15: 17, 16: 18, 17: 19, 18: 20, 19: 21, 20: 22, 21: 23, 22: 24, 23: 25, 24: 27, 25: 28, 26: 31, 27: 32, 28: 33, 29: 34, 30: 35, 31: 36, 32: 37, 33: 38, 34: 39, 35: 40, 36: 41, 37: 42, 38: 43, 39: 44, 40: 46,
                  41: 47, 42: 48, 43: 49, 44: 50, 45: 51, 46: 52, 47: 53, 48: 54, 49: 55, 50: 56, 51: 57, 52: 58, 53: 59, 54: 60, 55: 61, 56: 62, 57: 63, 58: 64, 59: 65, 60: 67, 61: 70, 62: 72, 63: 73, 64: 74, 65: 75, 66: 76, 67: 77, 68: 78, 69: 79, 70: 80, 71: 81, 72: 82, 73: 84, 74: 85, 75: 86, 76: 87, 77: 88, 78: 89, 79: 90}

In [3]:
# initialize new dataset with hard labels in coco format
coco = {
    "images": coco_gt["images"],
    "annotations": [], # initialize empty annotations list
    "categories": coco_gt["categories"]
}

annotation_id = 1

i = 0

for img_name, annotations in data["objects"].items():

    image_id = filename_to_id[img_name]

    coco["images"].append({
        "id": image_id,
        "file_name": img_name
    })

    for annot in annotations:

        bbox = annot["bbox"]
        soft_label = annot["soft_label"][:-1] # ignore background class (cant_solve)

        cls = int(np.argmax(soft_label))
        prob = soft_label[cls]

        if prob > soft_label_thresh:

            coco["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": id_map[cls],
                "bbox": [
                    bbox[0],
                    bbox[1],
                    bbox[2] - bbox[0],
                    bbox[3] - bbox[1]
                ],
                "area": (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]),
                "iscrowd": 0
            })

            annotation_id += 1
        
        i+=1


with open(f"../our_datasets/{dataset}_hard_labels_{soft_label_thresh}.json", "w") as f:
    json.dump(coco, f, indent=2)